# LECTURE 15 — DEMONSTRATION
### MODULE 6: DEEP LEARNING APPLICATIONS IN MACROECONOMICS

**DEMONSTRATION** — LSTM & GRU for Inflation Forecasting

*WAIFEM ML Facilitation Programme 2026 — Compatible with Google Colab & VS Code*

## ── SECTION 1: IMPORTS

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
sns.set_theme(style='whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'axes.titlesize': 11})

try:
    import google.colab          # noqa: F401
    print("Environment : Google Colab")
except ImportError:
    print("Environment : Local (VS Code / terminal)")

print(f"TensorFlow : {tf.__version__}")

## ── SECTION 2: DATA GENERATION

In [ ]:
def generate_inflation_series(n_obs: int = 300, seed: int = 42) -> pd.DataFrame:
    """
    Generate 300 monthly observations (25 years) with AR(1) inflation dynamics
    and five macroeconomic drivers.
    """
    rng   = np.random.default_rng(seed)
    dates = pd.date_range(start='1999-01-01', periods=n_obs, freq='MS')
    t     = np.linspace(0, 1, n_obs)

    money_growth  = 10.0 + np.cumsum(rng.normal(0, 0.10, n_obs)) + rng.normal(0, 0.6, n_obs)
    oil_price_chg = 8.0  * np.sin(6 * np.pi * t) + rng.normal(0, 3.5, n_obs)
    exchange_chg  = rng.normal(0, 1.5, n_obs)
    gdp_growth    = 3.5  + 1.2 * np.sin(8 * np.pi * t) + rng.normal(0, 0.4, n_obs)
    interest_rate = 7.5  + 2.0 * np.sin(4 * np.pi * t) + rng.normal(0, 0.4, n_obs)

    # AR(1) inflation
    inflation    = np.zeros(n_obs)
    inflation[0] = 5.0
    for i in range(1, n_obs):
        inflation[i] = (
            0.55 * inflation[i - 1]
            + 0.22 * money_growth[i]
            + 0.12 * oil_price_chg[i]
            + 0.14 * exchange_chg[i]
            - 0.18 * gdp_growth[i]
            - 0.08 * interest_rate[i]
            + rng.normal(0, 0.55)
        )

    return pd.DataFrame({
        'inflation':    inflation,
        'money_growth': money_growth,
        'oil_price_chg':oil_price_chg,
        'exchange_chg': exchange_chg,
        'gdp_growth':   gdp_growth,
        'interest_rate':interest_rate,
    }, index=dates)


df = generate_inflation_series()
FEATURE_COLS = ['money_growth', 'oil_price_chg', 'exchange_chg', 'gdp_growth', 'interest_rate']
TARGET_COL   = 'inflation'

print(f"Dataset : {df.shape[0]} monthly obs  x  {df.shape[1]} variables")
print(df.describe().round(2).to_string())

# Plot inflation series
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(df.index, df[TARGET_COL], color='steelblue', linewidth=1.5)
ax.set_title('Synthetic Inflation Series — Monthly (1999–2024)', fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Inflation Rate (%)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('lecture15_demo_inflation_series.png', bbox_inches='tight')
plt.show()
print("Saved : lecture15_demo_inflation_series.png")

## ── SECTION 3: PREPROCESSING & SEQUENCE CREATION

In [ ]:
LOOKBACK = 12     # 12-month look-back window
N_AHEAD  = 6      # months to forecast ahead (Section 8)

scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X_sc = scaler_X.fit_transform(df[FEATURE_COLS].values)
y_sc = scaler_y.fit_transform(df[[TARGET_COL]].values).ravel()


def make_sequences(X_data: np.ndarray, y_data: np.ndarray,
                   lookback: int) -> tuple:
    """
    Sliding-window sequence builder.

    Returns
    -------
    X_seq : ndarray, shape (n_samples, lookback, n_features)
    y_seq : ndarray, shape (n_samples,)
    """
    X_seq, y_seq = [], []
    for i in range(lookback, len(y_data)):
        X_seq.append(X_data[i - lookback: i, :])
        y_seq.append(y_data[i])
    return np.array(X_seq), np.array(y_seq)


X_all, y_all = make_sequences(X_sc, y_sc, LOOKBACK)
print(f"\nSequence shapes — X : {X_all.shape}  |  y : {y_all.shape}")

SPLIT       = int(0.80 * len(X_all))
X_train, X_test = X_all[:SPLIT], X_all[SPLIT:]
y_train, y_test = y_all[:SPLIT], y_all[SPLIT:]
print(f"Train : {X_train.shape[0]}  |  Test : {X_test.shape[0]}")

## ── SECTION 4: MODEL DEFINITIONS

In [ ]:
N_FEAT = X_all.shape[2]

def build_lstm(lookback: int, n_feat: int,
               units: int = 64, lr: float = 1e-3) -> tf.keras.Model:
    """Stacked LSTM — two layers with Dropout."""
    model = Sequential([
        LSTM(units,      return_sequences=True,  input_shape=(lookback, n_feat), name='lstm_1'),
        Dropout(0.20,    name='drop_1'),
        LSTM(units // 2, return_sequences=False,  name='lstm_2'),
        Dropout(0.20,    name='drop_2'),
        Dense(16, activation='relu', name='dense'),
        Dense(1,  activation='linear', name='output'),
    ], name='LSTM_Model')
    model.compile(optimizer=Adam(lr), loss='mse', metrics=['mae'])
    return model


def build_gru(lookback: int, n_feat: int,
              units: int = 64, lr: float = 1e-3) -> tf.keras.Model:
    """Stacked GRU — two layers with Dropout."""
    model = Sequential([
        GRU(units,      return_sequences=True,  input_shape=(lookback, n_feat), name='gru_1'),
        Dropout(0.20,   name='drop_1'),
        GRU(units // 2, return_sequences=False,  name='gru_2'),
        Dropout(0.20,   name='drop_2'),
        Dense(16, activation='relu', name='dense'),
        Dense(1,  activation='linear', name='output'),
    ], name='GRU_Model')
    model.compile(optimizer=Adam(lr), loss='mse', metrics=['mae'])
    return model


lstm_model = build_lstm(LOOKBACK, N_FEAT)
gru_model  = build_gru(LOOKBACK, N_FEAT)
lstm_model.summary()

## ── SECTION 5: TRAINING

In [ ]:
def make_es():
    return EarlyStopping(
        monitor='val_loss', patience=20,
        restore_best_weights=True, verbose=0,
    )

print("\nTraining LSTM ...")
h_lstm = lstm_model.fit(
    X_train, y_train, validation_split=0.15,
    epochs=200, batch_size=32, callbacks=[make_es()], verbose=0,
)

print("Training GRU  ...")
h_gru = gru_model.fit(
    X_train, y_train, validation_split=0.15,
    epochs=200, batch_size=32, callbacks=[make_es()], verbose=0,
)

print("Training done.")

## ── SECTION 6: LINEAR REGRESSION BASELINE

Flatten (samples, lookback, features) → (samples, lookback*features)

In [ ]:
X_tr_flat = X_train.reshape(X_train.shape[0], -1)
X_te_flat = X_test.reshape(X_test.shape[0], -1)

lr_model = LinearRegression()
lr_model.fit(X_tr_flat, y_train)

## ── SECTION 7: EVALUATE & COMPARE

In [ ]:
def inv_y(arr: np.ndarray) -> np.ndarray:
    """Inverse-transform scaled inflation values."""
    return scaler_y.inverse_transform(arr.reshape(-1, 1)).ravel()


y_actual    = inv_y(y_test)
y_pred_lstm = inv_y(lstm_model.predict(X_test, verbose=0).ravel())
y_pred_gru  = inv_y(gru_model.predict(X_test,  verbose=0).ravel())
y_pred_lr   = inv_y(lr_model.predict(X_te_flat).ravel())


def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    return {
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'MAE':  mean_absolute_error(y_true, y_pred),
        'R²':   r2_score(y_true, y_pred),
    }


results = pd.DataFrame({
    'LSTM':               compute_metrics(y_actual, y_pred_lstm),
    'GRU':                compute_metrics(y_actual, y_pred_gru),
    'Linear Regression':  compute_metrics(y_actual, y_pred_lr),
}).T

print("\n── Model Comparison — Test Set ──────────────────────────────────────")
print(results.round(4).to_string())

## ── SECTION 8: LEARNING CURVES

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Learning Curves — LSTM vs GRU', fontsize=12, fontweight='bold')

for ax, hist, name, c in zip(
    axes,
    [h_lstm, h_gru],
    ['LSTM', 'GRU'],
    ['steelblue', 'seagreen'],
):
    ax.plot(hist.history['loss'],     label='Train',      color=c,       linewidth=1.5)
    ax.plot(hist.history['val_loss'], label='Validation', color='tomato', linewidth=1.5, linestyle='--')
    ax.set_title(f'{name} — Loss (MSE)')
    ax.set_xlabel('Epoch')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('lecture15_demo_learning_curves.png', bbox_inches='tight')
plt.show()
print("Saved : lecture15_demo_learning_curves.png")

## ── SECTION 9: PREDICTION COMPARISON PLOT

In [ ]:
test_dates = df.index[LOOKBACK + SPLIT:]

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(test_dates, y_actual,    label='Actual',                           color='black',     linewidth=2.0)
ax.plot(test_dates, y_pred_lstm, label=f'LSTM  (R²={results.loc["LSTM","R²"]:.3f})',          color='steelblue', linewidth=1.8, linestyle='--')
ax.plot(test_dates, y_pred_gru,  label=f'GRU   (R²={results.loc["GRU","R²"]:.3f})',           color='seagreen',  linewidth=1.8, linestyle='-.')
ax.plot(test_dates, y_pred_lr,   label=f'Linear Regression (R²={results.loc["Linear Regression","R²"]:.3f})',
        color='orange', linewidth=1.5, linestyle=':')
ax.set_title('Inflation Forecast — LSTM vs GRU vs Linear Regression (Out-of-Sample)',
             fontsize=11, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Inflation Rate (%)')
ax.legend()
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.savefig('lecture15_demo_model_comparison.png', bbox_inches='tight')
plt.show()
print("Saved : lecture15_demo_model_comparison.png")

## ── SECTION 10: MULTI-STEP FORECASTING (6 months ahead)

Strategy : train a direct multi-output LSTM that predicts the next N_AHEAD

inflation values simultaneously (avoids error accumulation of recursive methods).

In [ ]:
print(f"\nBuilding direct {N_AHEAD}-step forecasting model ...")


def build_lstm_multistep(lookback: int, n_feat: int,
                         n_ahead: int, units: int = 64) -> tf.keras.Model:
    model = Sequential([
        LSTM(units,      return_sequences=True,  input_shape=(lookback, n_feat), name='lstm_1'),
        Dropout(0.20,    name='drop_1'),
        LSTM(units // 2, return_sequences=False,  name='lstm_2'),
        Dropout(0.20,    name='drop_2'),
        Dense(32, activation='relu', name='dense'),
        Dense(n_ahead,   name='output'),          # N_AHEAD outputs
    ], name='LSTM_MultiStep')
    model.compile(optimizer=Adam(1e-3), loss='mse')
    return model


def make_multistep_sequences(y_data: np.ndarray, X_data: np.ndarray,
                              lookback: int, n_ahead: int) -> tuple:
    """Sequences where y contains the next n_ahead target values."""
    X_seq, y_seq = [], []
    for i in range(lookback, len(y_data) - n_ahead + 1):
        X_seq.append(X_data[i - lookback: i, :])
        y_seq.append(y_data[i: i + n_ahead])
    return np.array(X_seq), np.array(y_seq)


X_ms, y_ms = make_multistep_sequences(y_sc, X_sc, LOOKBACK, N_AHEAD)
split_ms    = int(0.80 * len(X_ms))
X_tr_ms, X_te_ms = X_ms[:split_ms], X_ms[split_ms:]
y_tr_ms, y_te_ms = y_ms[:split_ms], y_ms[split_ms:]

ms_model = build_lstm_multistep(LOOKBACK, N_FEAT, N_AHEAD)
ms_model.fit(
    X_tr_ms, y_tr_ms, validation_split=0.15,
    epochs=200, batch_size=32,
    callbacks=[EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True, verbose=0)],
    verbose=0,
)
print("Multi-step model trained.")

# Forecast on the last available sequence
last_X     = X_sc[-LOOKBACK:].reshape(1, LOOKBACK, N_FEAT)
ms_pred_sc = ms_model.predict(last_X, verbose=0).ravel()
ms_pred    = scaler_y.inverse_transform(ms_pred_sc.reshape(-1, 1)).ravel()

forecast_dates = pd.date_range(
    start=df.index[-1] + pd.DateOffset(months=1),
    periods=N_AHEAD, freq='MS',
)

fig, ax = plt.subplots(figsize=(13, 5))
hist_window = 36
ax.plot(df.index[-hist_window:], df[TARGET_COL].iloc[-hist_window:],
        color='steelblue', linewidth=2, label='Historical Inflation')
ax.plot(forecast_dates, ms_pred,
        color='tomato', linewidth=2, linestyle='--', marker='o', markersize=5,
        label=f'LSTM {N_AHEAD}-Month Forecast')
ax.fill_between(forecast_dates,
                ms_pred * 0.90, ms_pred * 1.10,
                alpha=0.20, color='tomato', label='±10 % uncertainty band')
ax.axvline(df.index[-1], color='grey', linestyle=':', linewidth=1.2)
ax.set_title(f'LSTM — Direct {N_AHEAD}-Month-Ahead Inflation Forecast',
             fontsize=11, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Inflation Rate (%)')
ax.legend()
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.savefig('lecture15_demo_multistep_forecast.png', bbox_inches='tight')
plt.show()
print("Saved : lecture15_demo_multistep_forecast.png")

print("\nForecast values (next 6 months):")
for d, v in zip(forecast_dates, ms_pred):
    print(f"  {d.strftime('%b %Y')} : {v:.2f} %")

## ── SECTION 11: SUMMARY

In [ ]:
print("""
╔═══════════════════════════════════════════════════════════════════╗
║  LECTURE 15 DEMO COMPLETE                                         ║
╠═══════════════════════════════════════════════════════════════════╣
║  Concepts demonstrated:                                           ║
║   1. AR(1) inflation series with macroeconomic drivers            ║
║   2. Sliding-window sequence construction (lookback=12)           ║
║   3. Stacked LSTM architecture — 2 layers + Dropout               ║
║   4. Stacked GRU  architecture — 2 layers + Dropout               ║
║   5. Linear regression baseline for comparison                    ║
║   6. RMSE / MAE / R² model comparison table                       ║
║   7. Direct multi-output LSTM for 6-step-ahead forecasting        ║
╠═══════════════════════════════════════════════════════════════════╣
║  Saved outputs:                                                   ║
║   lecture15_demo_inflation_series.png                             ║
║   lecture15_demo_learning_curves.png                              ║
║   lecture15_demo_model_comparison.png                             ║
║   lecture15_demo_multistep_forecast.png                           ║
╚═══════════════════════════════════════════════════════════════════╝
""")